In [ ]:
# Cell 1: Colab Environment Setup
import os

if 'google.colab' in str(get_ipython()):
    REPO_NAME = "ICELLI-2026-Talk"
    REPO_URL = "https://github.com/codewithbello/ICELLI-2026-Talk.git"
    
    # 1. Clone repository if not already present
    if not os.path.exists(f"/content/{REPO_NAME}"):
        !git clone {REPO_URL}
        
    # 2. Set working directory to repo folder
    %cd /content/{REPO_NAME}
    print(f"✅ Ready! Working directory: {os.getcwd()}")


In [ ]:
"""
=====================================================================================================================
ICELLI 2026: TIME SERIES ANALYSIS IN SPACE PHYSICS: A PRACTICAL PYTHON WORKFLOW FOR IONOSPHERIC AND GEOMAGNETIC DATA
=====================================================================================================================

Presenter          : Bello Saeed Abioye
Affiliation        : Department of Physics, University of Ilorin
Email              : bello.sa@unilorin.edu.ng
ORCiD              : https://orcid.org/0000-0002-5527-1110

Key Learning Resources:
  - Python Official Documentation : https://www.python.org/
  - Pandas Getting Started Guides : https://pandas.pydata.org/docs/getting_started/intro_tutorials/index.html
  - Markdown Cheat Sheet Guide    : https://www.markdownguide.org/cheat-sheet/
  - IGS Network Station Portal    : https://network.igs.org/
  - CDDIS NASA RINEX Data Archive : https://cddis.nasa.gov/archive/gnss/data/daily/
  - Dr. Gopi Seemala GPS-TEC Tool : https://seemala.blogspot.com/

Station Information:
  - Station Name     : Toro, Nigeria (IGS Code: CGGN)
  - Coordinates      : Latitude: 10.123° N | Longitude: 9.118° E | Elevation: 916.7 m
  - Receiver         : JAVAD TRE_G3TH DELTA
  - Antenna          : ASH701945B_M
  - Constellations   : GPS + GLONASS

Data Pipeline Overview:
  1. IGS Network     : Locate station CGGN, inspect operational status and data availability.
  2. CDDIS Archive   : Download daily RINEX observation files (e.g., cggn0010.14d.Z).
  3. GOPI Software   : Uncompress and process RINEX files to derive slant and vertical TEC (.Cmn).
  4. Python / Pandas : Ingest, concatenate, clean, aggregate hourly diurnal cycles, and visualize.

Key GNSS-TEC Output Parameters:
  - Jdatet   : Modified Julian Date / Epoch timestamp
  - Time     : Universal Time (UTC) in decimal hours (0.000 to 24.000 UT)
  - PRN      : Pseudo-Random Noise number (Satellite identifier)
  - Az       : Satellite Azimuth angle (degrees)
  - Ele      : Satellite Elevation angle (degrees)
  - Lat, Lon : Ionospheric Pierce Point (IPP) coordinates
  - Stec     : Slant Total Electron Content along signal ray path (TECU)
  - Vtec     : Vertical Total Electron Content mapped to zenith (TECU)
  - S4       : Amplitude scintillation index (dimensionless, -99 indicates unavailable)
========================================================================================
"""

# ICELLI 2026: GNSS Total Electron Content (TEC) Analysis in Python
**Presenter:** Bello Saeed Abioye  
**Affiliation:** Department of Physics, University of Ilorin | [bello.sa@unilorin.edu.ng](mailto:bello.sa@unilorin.edu.ng)  
**ORCiD:** [https://orcid.org/0000-0002-5527-1110](https://orcid.org/0000-0002-5527-1110)  

---

### 🛰️ CGGN GNSS Station Overview (Toro, Nigeria)

![CGGN GNSS Receiver Station](CGGN-20120704-NW.JPG)

- **Station Code:** `CGGN` (Toro, Bauchi State, Nigeria)
- **Geographic Coordinates:** $10.123^\circ\text{ N}, 9.118^\circ\text{ E}$ (Elevation: $916.7\text{ m}$)
- **Instrumentation:** JAVAD TRE_G3TH DELTA Receiver with ASH701945B_M Antenna

---

### 📚 Workflow Summary
1. **Data Acquisition:** Daily RINEX observations from [NASA CDDIS](https://cddis.nasa.gov/archive/gnss/data/daily/).
2. **Pre-processing:** Processed via [Seemala GPS-TEC analysis tool](https://seemala.blogspot.com/) to produce `.Cmn` ASCII text files.
3. **Analysis:** Batch load multi-day observations, calculate diurnal (hourly) variations, and generate publication-ready plots.

### Section 1: Importing Core Scientific Libraries & Setting Visual Styles
- **`os`**: File system navigation and batch directory listing.
- **`pandas`**: High-performance data structures, CSV parsing, and groupby aggregation.
- **`numpy`**: Mathematical operations and array transformations.
- **`matplotlib.pyplot`**: 2D plotting, subplots, and publication-ready formatting.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Set modern scientific aesthetic for all figures
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 11
plt.rcParams['figure.titlesize'] = 14

print("Core libraries imported successfully!")

### Section 2: Reading Single-Day GNSS TEC Data (`.Cmn` Format)
TEC files produced by the Seemala GPS-TEC software contain:
1. **Header Lines**: 4 metadata lines containing station name, coordinates, and column descriptions.
2. **Whitespace Delimitation**: Columns separated by variable spaces (`sep=r'\s+'`).
3. **Missing Value Encoding**: Unobserved/invalid data represented as `-99.0000` (e.g. `S4` index).

In [ ]:
# Define explicit column names matching the GOPI output format
columns = ['Jdatet', 'Time', 'PRN', 'Az', 'Ele', 'Lat', 'Lon', 'Stec', 'Vtec', 'S4']

# Path to sample single-day file (Day 001 / Jan 01, 2014)
single_file = 'CGGN-2014_TEC_Data/cggn001-2014-01-01.Cmn'

# Ingest single-day file
df_single = pd.read_csv(
    single_file,
    sep=r'\s+',
    header=4,
    na_values=-99.0000,
    names=columns
)

# Display data overview
print("--- Single Day Data Summary (CGGN Day 001, 2014) ---")
print(f"Total Observations: {len(df_single):,}")
print("\nFirst 5 Records:")
print(df_single.head())
print("\nSummary Statistics for VTEC:")
print(df_single['Vtec'].describe())

### Section 3: Batch Loading and Combining Multiple Days of TEC Data
Rather than analyzing one file at a time, we dynamically discover all `.Cmn` daily files
in the directory and concatenate them into a single consolidated `DataFrame`.

In [ ]:
data_dir = r"./CGGN-2014_TEC_Data"

# 1. Discover and sort all daily .Cmn files in the directory
# Rather than analyzing one file at a time, we dynamically discover all `.Cmn` daily files
# in the directory
files = sorted([f for f in os.listdir(data_dir) if f.endswith('.Cmn')])
print(f"Discovered {len(files)} daily TEC files:")
for f in files[:5]:
    print(f"  -> {f}")
if len(files) > 5:
    print(f"  ... and {len(files) - 5} more files.")

2. Iterate through all files and collect DataFrames
Concatenate the single '*.Cmn' files into a single consolidated `DataFrame`.

all_dataframes = []
for f in files:
    file_path = os.path.join(data_dir, f)
    day_df = pd.read_csv(
        file_path,
        sep=r'\s+',
        header=4,
        na_values=-99.0000,
        names=columns
    )
    all_dataframes.append(day_df)

3. Concatenate into a single master DataFrame
df = pd.concat(all_dataframes, ignore_index=True)
print(f"\n--- Combined Dataset Summary ---")
print(f"Total records combined: {len(df):,}")
print(df.head())

In [ ]:
# 4. Save combined dataset to CSV for fast future loading
csv_output_file = 'cggn_2014_combined_tec.csv'
df.to_csv(csv_output_file, index=False)
print(f"Combined data saved to '{csv_output_file}' successfully!")

# To reload the combined dataset later without reprocessing:
# df = pd.read_csv('cggn_2014_combined_tec.csv')

### Section 4: Data Cleaning & Hourly Aggregation
- **Data Cleaning**: Remove rows with missing (`NaN`) `Vtec` or `Time` values.
- **Hourly Binning**: Convert fractional decimal hours (e.g. `7.683` $\rightarrow$ `7`) to integer hour bins ($0 \le \text{Hour} \le 23$).
- **Statistical Aggregation**: Compute hourly mean ($\mu$), median, and standard deviation ($\sigma$) to trace the diurnal cycle.

In [ ]:
# Drop missing values
clean_df = df.dropna(subset=['Time', 'Vtec']).copy()

# Extract integer hour bin (0 to 23 UT)
clean_df['Hour_int'] = clean_df['Time'].apply(np.floor).astype(int)

# Group by hour and calculate diurnal statistics
hourly_stats = clean_df.groupby('Hour_int')['Vtec'].agg(
    mean='mean',
    median='median',
    std='std',
    count='count'
).reset_index()

print("--- Hourly VTEC Diurnal Statistics (0–23 UT) ---")
print(hourly_stats)

### Section 5: Publication-Quality Diurnal Variation Visualizations
In GNSS ionospheric studies, diurnal VTEC variation is visualized by overlaying:
1. **Raw Observation Points** (represented in transparent gray to display scatter and variance).
2. **Hourly Mean Curve** (bold curve tracing the diurnal ionospheric peak and nighttime minimum).
3. **Variability Band ($\pm 1\sigma$)** (shaded envelope representing geomagnetic/day-to-day variability).

In [ ]:
plt.figure(figsize=(12, 6))

# 1. Raw scatter data points (gray with low opacity)
plt.scatter(
    clean_df['Time'], clean_df['Vtec'],
    color='gray', alpha=0.15, s=6,
    label='Raw Observations'
)

# 2. Hourly Mean curve
plt.plot(
    hourly_stats['Hour_int'], hourly_stats['mean'],
    color='red', marker='o', linewidth=2.5,
    label='Hourly Mean VTEC'
)

# 3. Variability (+/- 1 standard deviation)
plt.fill_between(
    hourly_stats['Hour_int'],
    hourly_stats['mean'] - hourly_stats['std'],
    hourly_stats['mean'] + hourly_stats['std'],
    color='red', alpha=0.2,
    label=r'Variability ($\pm 1\sigma$)'
)

# Titles and Axis formatting
plt.title("Diurnal Variation of Vertical Total Electron Content (VTEC) — CGGN (Toro, Nigeria)", fontsize=14, fontweight='bold')
plt.xlabel("Time (UT)", fontsize=12, fontweight='bold')
plt.ylabel("VTEC (TECU)", fontsize=12, fontweight='bold')
plt.xlim(0, 24)
plt.xticks(range(0, 25, 2))
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='upper right', frameon=True)
plt.tight_layout()
plt.show()

### Section 6: Side-by-Side Comparison (Raw vs. Diurnal Profile)
A clean 2-panel figure comparing:
- **Panel (a)**: Raw scatter of all satellite passes over the month.
- **Panel (b)**: Mean diurnal cycle with standard error / variability bands.

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(15, 6), sharey=True)

# Panel 1: Raw Observations
axes[0].scatter(clean_df['Time'], clean_df['Vtec'], color='tab:blue', alpha=0.12, s=5)
axes[0].set_title("(a) VTEC Observations", fontweight='bold')
axes[0].set_xlabel("Time (UT)", fontweight='bold')
axes[0].set_ylabel("VTEC (TECU)", fontweight='bold')
axes[0].set_xlim(0, 24)
axes[0].set_xticks(range(0, 25, 2))
axes[0].grid(True, linestyle='--', alpha=0.6)

# Panel 2: Hourly Averaged Diurnal Profile
axes[1].plot(hourly_stats['Hour_int'], hourly_stats['mean'], color='tab:red', marker='o', lw=2.2, label='Hourly Mean')
axes[1].plot(hourly_stats['Hour_int'], hourly_stats['median'], color='tab:green', linestyle='--', marker='s', lw=1.8, label='Hourly Median')
axes[1].fill_between(
    hourly_stats['Hour_int'],
    hourly_stats['mean'] - hourly_stats['std'],
    hourly_stats['mean'] + hourly_stats['std'],
    color='tab:red', alpha=0.18, label=r'$\pm 1\sigma$ Range'
)
axes[1].set_title("(b) Mean Diurnal VTEC", fontweight='bold')
axes[1].set_xlabel("Time (UT)", fontweight='bold')
axes[1].set_xlim(0, 24)
axes[1].set_xticks(range(0, 25, 2))
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend(loc='upper right', frameon=True)

plt.suptitle("GNSS Ionospheric Analysis — CGGN Station (January 2014)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n--- GNSS TEC Tutorial Script Completed Successfully! ---")